In [1]:

                        
!pip install -q transformers           # For Whisper + LLaMA
!pip install -q accelerate             # Helps load big models efficiently
!pip install -q bitsandbytes           # For quantization (makes LLaMA smaller)
!pip install -q openai-whisper         # OpenAI's original Whisper library
!pip install -q sentencepiece          # Tokenizer for LLaMA
!pip install -q datasets               # To get sample audio for testing
!pip install -q soundfile              # Audio file handling
!pip install -q librosa                # Audio processing
!pip install -q pydub                  # Audio format conversion

print(" All libraries installed!")

 All libraries installed!


# CELL 2 — Check Your GPU

In [2]:

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU found: {gpu_name}")
    print(f" GPU Memory: {gpu_mem:.1f} GB")
else:
    print(" No GPU found. Go to Runtime → Change runtime type → T4 GPU")

# Set device globally
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n Using device: {DEVICE}")

 GPU found: Tesla T4
 GPU Memory: 15.6 GB

 Using device: cuda


#  CELL 3 — Load Whisper (The Transcription Model)

In [3]:

import whisper

print(" Loading Whisper model... ")

whisper_model = whisper.load_model("small", device=DEVICE)

print(" Whisper loaded successfully!")
print(f"   Model is running on: {DEVICE}")

 Loading Whisper model... (downloads ~500MB first time)
 Whisper loaded successfully!
   Model is running on: cuda


# CELL 4 — Load Quantized LLaMA (The Reasoning Model)

In [4]:

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# --- Step A: Define quantization settings ---
# This tells the library: "load this model in 4-bit mode"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # Use 4-bit quantization
    bnb_4bit_compute_dtype=torch.float16,    # Do math in float16 (faster)
    bnb_4bit_use_double_quant=True,          # Extra compression
    bnb_4bit_quant_type="nf4"               # Best quality 4-bit format
)

# --- Step B: Choose your model ---

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"



print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f" Loading quantized LLM...")
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",       # Automatically puts model on GPU
    trust_remote_code=True
)

print(" Quantized LLM loaded!")



Loading tokenizer for TinyLlama/TinyLlama-1.1B-Chat-v1.0...


 Loading quantized LLM...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 Quantized LLM loaded!


# CELL 5 — Get a Sample Audio File to Test

In [12]:
import os

# ← This is your ACTUAL correct path (found from the scan above)
AUDIO_FILE = "/kaggle/input/datasets/likinpark/urdudataset/urdu.m4a"

# Verify it
if os.path.exists(AUDIO_FILE):
    print(f" File found!")
    print(f"    Path : {AUDIO_FILE}")
else:
    print("Still not found")

 File found!
    Path : /kaggle/input/datasets/likinpark/urdudataset/urdu.m4a


# # Convert .m4a → .wav for Whisper

In [14]:

!apt-get install -q ffmpeg

from pydub import AudioSegment

audio = AudioSegment.from_file(
    "/kaggle/input/datasets/likinpark/urdudataset/urdu.m4a",  # ← correct path
    format="m4a"
)
audio.export("/kaggle/working/urdu.wav", format="wav")

AUDIO_FILE = "/kaggle/working/urdu.wav"   # ← use this going forward
print(f" Converted successfully!")


Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
 Converted successfully!


# # CELL 6 — Define the transcribe_audio function

In [ ]:
import whisper

# Make sure whisper model is loaded
print("Loading Whisper model...")
whisper_model = whisper.load_model("small", device=DEVICE)
print(" Whisper loaded!")

# Define the function
def transcribe_audio(audio_path, model=whisper_model):
    print(f" Transcribing: {audio_path}")
    
    result = whisper_model.transcribe(
        audio_path,
        fp16=(DEVICE == "cuda"),
        language="ur",        # ← Urdu
        verbose=False
    )
    
    transcription = result["text"].strip()
    print(f" Done! Transcribed text:\n{transcription}")
    
    return transcription

print("transcribe_audio function is ready!")


Loading Whisper model...
 Whisper loaded!
transcribe_audio function is ready!


# NEXT: Cell 6 — Transcribe Your Urdu Audio

In [17]:

# AUDIO_FILE is already set to "/kaggle/working/urdu.wav"

transcribed_text = transcribe_audio(AUDIO_FILE)

print("\n📝 Transcribed Text:")
print(transcribed_text)

🎤 Transcribing: /kaggle/working/urdu.wav


100%|██████████| 1973/1973 [00:07<00:00, 270.86frames/s]

 Done! Transcribed text:
اللم کیا تو مجھے ایک سپیچ کو کسی کنورٹ کر سکتے ہیں؟ ایک اللم کیا تکنیق سیوز کرتے ہیں کہ وہ ایک اوڈیو سپیچ ہوتی ہے اس کو ٹرانسکرائپ کر دیتا ہے انٹو ٹکس میں

📝 Transcribed Text:
اللم کیا تو مجھے ایک سپیچ کو کسی کنورٹ کر سکتے ہیں؟ ایک اللم کیا تکنیق سیوز کرتے ہیں کہ وہ ایک اوڈیو سپیچ ہوتی ہے اس کو ٹرانسکرائپ کر دیتا ہے انٹو ٹکس میں


# CELL 7 — Pass Transcription to LLM for Reasoning

In [26]:

# The LLM receives the transcribed question and generates
# a thoughtful, reasoned answer.


def ask_llm(question, model=llm_model, tok=tokenizer, max_new_tokens=300):
    """
    Takes a question (text) and returns the LLM's answer.
    
    Args:
        question: the transcribed text from Whisper
        model: loaded quantized LLM
        tok: tokenizer
        max_new_tokens: how long the answer can be
    
    Returns:
        answer string
    """

    print(f"\n Sending to LLM for reasoning...")
    print(f"   Question: '{question}'")

    # --- Step A: Format the prompt ---

    prompt = f"""<|system|>
You are a helpful, clear and concise AI assistant. 
Answer the question based on logical reasoning.
Provide a well-structured answer.</s>
<|user|>
{question}</s>
<|assistant|>
"""

    # --- Step B: Tokenize the prompt ---
    # Convert text → numbers the model understands
    inputs = tok(
        prompt,
        return_tensors="pt",      # Return PyTorch tensors
        truncation=True,          # Cut if too long
        max_length=512            # Max input length
    ).to(DEVICE)                  # Send to GPU

    print(f"   Input token count: {inputs['input_ids'].shape[1]}")

    # --- Step C: Generate the answer ---
    # This is where the LLM "thinks" and writes its response
    with torch.no_grad():         # Don't calculate gradients (saves memory)
        output_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,   # Max answer length
            do_sample=True,                  # Add some creativity
            temperature=0.7,                 # 0=robotic, 1=creative
            top_p=0.9,                       # Nucleus sampling
            repetition_penalty=1.1,          # Avoid repeating words
            pad_token_id=tok.eos_token_id
        )

    # --- Step D: Decode back to text ---
    # Convert numbers → readable text
    # Only take the NEW tokens (not the input prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tok.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"   Output token count: {len(new_tokens)}")

    return answer


llm_answer = ask_llm(transcribed_text)

print("\n" + "="*60)
print(" LLM ANSWER:")
print("="*60)
print(llm_answer)
print("="*60)    


 Sending to LLM for reasoning...
   Question: 'اللم کیا تو مجھے ایک سپیچ کو کسی کنورٹ کر سکتے ہیں؟ ایک اللم کیا تکنیق سیوز کرتے ہیں کہ وہ ایک اوڈیو سپیچ ہوتی ہے اس کو ٹرانسکرائپ کر دیتا ہے انٹو ٹکس میں'
   Input token count: 220
   Output token count: 300

 LLM ANSWER:
نا کیا تو مجھے ایک سپیچ کو کسی کنورٹ کر سکتے ہیں۔ به اس کا اعتمدا ہے کہ فراہم کردی گئی کہ ایک اوڈیو سپیچ ہوتی ہے اس کو ٹرانسکرائپ کر دیتا ہے انٹو ٹکس میں، اس کا سنگ کی کسی کنورٹ کر دیتا ہے اس کو کسی کنورٹ کر دیتا ہے اس کو ٹرانسکرائپ کر دیتا ہے انٹو ٹکس میں، اس کا سنگ کی کسی کنورٹ کر د
